# 11 — Next-interval direction vs. volatility-adjusted targets

**Development-only experiment.** Compare three label definitions using **five expanding chronological folds**. The newest **20% of timestamps is locked away**. No random train/test splitting. No random forests, regressors or neural networks.

**Interpretation:** Accuracy across *different targets* is **not directly comparable**; use target-specific prior baselines, balanced accuracy, ROC-AUC, class prevalence and confusion matrices. Volatility-adjusted labels use the **current bar's trailing 20-bar volatility**, never next-bar volatility. Binary classification excludes exactly zero next returns.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if not (ROOT / 'src' / 'stock_ml_dynamic').exists():
    raise FileNotFoundError(f'Run from the repository root or notebooks directory: {ROOT}')
sys.path.insert(0, str(ROOT / 'src'))
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.target_diagnostics import (run_target_diagnostics, summary_targets, plot_target_diagnostics)

candidates = [ROOT / 'artifacts/master_tables/1d', ROOT / 'notebooks/artifacts/master_tables/1d']
required = ('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
master_dir = next((p for p in candidates if all((p / f).exists() for f in required)), None)
if master_dir is None:
    raise FileNotFoundError('Master artifacts not found. Run Notebook 03 first. Checked: ' + ', '.join(map(str,candidates)))
master, latest, bundle, manifest = load_master(master_dir)
print('Master directory:',master_dir)
print('Master rows:',len(master),'symbols:',master.symbol.nunique())


In [ ]:
# Main experiment: 5 folds × 3 targets × 3 models = 45 fits.
# Each fold re-fits the target cutoffs (where applicable), K-means,
# original feature preprocessing, and added-feature scaler using training data only.
OUTPUT = ROOT / 'artifacts/target_diagnostics/1d'
N_FOLDS = 5
MAX_TRAIN_ROWS = 120_000
VOLATILITY_MULTIPLIER = 0.5   # Fixed in advance; do not tune against final holdout

scores, roc, prevalence, confusion, eligibility, holdout_boundary = run_target_diagnostics(
    master, n_folds=N_FOLDS, holdout_fraction=0.20,
    max_train_rows=MAX_TRAIN_ROWS, selected_k=4, cluster_fit_limit=15_000,
    volatility_multiplier=VOLATILITY_MULTIPLIER, output_dir=OUTPUT,
)
print('Locked development boundary:',holdout_boundary)
print('Holdout status: NOT EVALUATED')


In [ ]:
summary = summary_targets(scores)
display(summary)
display(scores[['fold','target','model','accuracy','balanced_accuracy','roc_auc','log_loss','fit_seconds']])


In [ ]:
display(prevalence.pivot_table(index=['fold','target'],columns='class_id',values='test_share'))
display(confusion.groupby(['target','model','actual','predicted'],as_index=False)['count'].sum().head(45))
print('Binary zero-return exclusions (test):')
display(scores[scores.target.eq('binary_direction')].groupby('fold',as_index=False).zero_test_excluded.max())


In [ ]:
summary = plot_target_diagnostics(scores,roc,prevalence,OUTPUT)
print('Saved results to:',OUTPUT)


## Interpretation and next decision

- **Tertiles**: down=2, middle=0, up=1. Boundaries fitted to each fold's training next returns only.
- **Volatility-adjusted**: down if next return < −0.5 × *current* trailing 20-bar return volatility; up if > +0.5 × volatility; otherwise neutral.
- **Binary**: down=0, up=1; exact zero returns excluded (counts reported).
- Compare each model to its **own target's baseline** and inspect fold-to-fold stability. A higher binary accuracy does not necessarily mean a more profitable or more informative model.
- No final holdout results are generated. Keep that period untouched until the target and model are frozen.
